In [9]:
import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json


# set directories
WD_junxi = Path('PATH_TO_DATA')
WD_junxi_win = Path("PATH_TO_DATA")

WD = WD_junxi

data_dir = Path(WD/'EntTemplates/Analysis/python_Patent/data/')
invariant_data_dir = Path(WD/'EntTemplates/Analysis/Stata/invariant_data/')
output_dir = Path(WD/'EntTemplates/Analysis/python_BERT/data_patent/')
data_v2 = Path(WD/'EntTemplates/Analysis/Stata/data_v2/')
output_dir_matching = Path(WD/'EntTemplates/Analysis/python_Patent/output/')

# 1. performances

In [28]:
# Function to read and concatenate JSON files
def concat_test_results(root_dir=output_dir/'model'):
    results = {}
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file == 'test_results.json':
                with open(os.path.join(subdir, file), 'r') as f:
                    try:
                        data = json.load(f)
                        results[subdir.split('-')[-1]] = data
                    except json.JSONDecodeError:
                        print(f"Error decoding JSON from file: {os.path.join(subdir, file)}")
    return results

# Execute the function
test_results = concat_test_results()
df_eval = pd.DataFrame(test_results).T
df_eval_mean = df_eval.mean(axis=0)

# 2. results

In [10]:
def concat_positive_files(root_dir=output_dir/'positive'):
    file_list = []
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith('positive_bert.csv'):
                new_df = pd.read_csv(os.path.join(subdir, file))
                new_df['file'] = file.split('_')[0]
                new_df = new_df[new_df['positive'] == 1]
                file_list.append(new_df)
                
    return pd.concat(file_list, axis=0, ignore_index=True)
df_results = concat_positive_files()

### Assignee
g_assignee = pd.read_csv(data_dir / 'g_assignee_disambiguated.tsv', sep='\t',low_memory=False)
# keep if assignee_sequence is 0
g_assignee = g_assignee[g_assignee['assignee_sequence'] == 0]
# drop if disambig_assignee_organization is empty
g_assignee = g_assignee[g_assignee['disambig_assignee_organization'].notnull()]
# keep 'assignee_sequence', 'assignee_id','disambig_assignee_organization', 'assignee_type', 'location_id'
g_assignee = g_assignee[['patent_id','assignee_id','disambig_assignee_organization', 'location_id']]

### Patent
g_patent = pd.read_csv(data_dir / 'g_patent.tsv', sep='\t',low_memory=False) 
# keep 'patent_id', 'patent_type', 'patent_date', 'wipo_kind'
g_patent = g_patent[['patent_id', 'patent_type', 'patent_date']]
# keep if patent_type is 'utility'
g_patent = g_patent[g_patent['patent_type'] == 'utility']
# get patent year from patent dat
g_patent['patent_year'] = g_patent['patent_date'].str[:4]
# make int
g_patent['patent_year'] = g_patent['patent_year'].astype(int, errors='ignore')
# keep if patent_year is 2000 onwards
g_patent = g_patent[g_patent['patent_year'] >= 2000]

### Location
g_location = pd.read_csv(data_dir / 'g_location_disambiguated.tsv', sep='\t',low_memory=False)

### generate patent_assignee_location
g_patent_assignee = pd.merge(g_patent, g_assignee, on='patent_id', how='inner')
# Merge in location
g_patent_assignee_location = pd.merge(g_patent_assignee, g_location, on='location_id', how='inner')

# make patent_id string
df_results['patent_id'] = df_results['patent_id'].astype(str)
g_patent_assignee_location['patent_id'] = g_patent_assignee_location['patent_id'].astype(str)
df_results_final = pd.merge(df_results, g_patent_assignee_location[['patent_id','disambig_country','patent_date']], on='patent_id', how='left')
# drop Unnamed: 5
df_results_final = df_results_final.drop(columns='Unnamed: 5')

count_df_results = df_results_final.groupby('file').size().reset_index(name='counts')
count_df_results = count_df_results.merge(df_eval, how='left', left_on='file', right_index=True)

In [11]:
### GET TRAINING DATA

# read in patent data
g_patent = pd.read_csv(data_dir/'g_patent.tsv', sep='\t', low_memory=False) 
g_patent = g_patent[g_patent['patent_type'] == 'utility']

g_assignee = pd.read_csv(data_dir/'g_assignee_disambiguated.tsv', sep='\t', low_memory=False)
g_assignee = g_assignee[g_assignee['assignee_sequence'] == 0]
g_assignee = g_assignee[['patent_id', 'assignee_id', 'disambig_assignee_organization', 'location_id']]
g_patent = g_patent.merge(g_assignee, on='patent_id', how='inner')
del g_assignee

g_location = pd.read_csv(data_dir/'g_location_disambiguated.tsv', sep='\t', low_memory=False)
g_patent = g_patent.merge(g_location[['location_id', 'disambig_country']], on='location_id', how='left')

pb_assignee = pd.read_excel(output_dir/'matched_result.xlsx')

pb_marketmap = pd.read_csv(output_dir/'predicted_positive_v2.csv')
pb_marketmap.sort_values(by=['companyid'], inplace=True)

pb_marketmap['count_sector'] = pb_marketmap.groupby('companyid')['companyid'].transform('count')
pb_marketmap['unique_segment'] = pb_marketmap['marketmap'] + pb_marketmap['segment']
pb_marketmap['count_segment'] = pb_marketmap.groupby('companyid')['unique_segment'].transform('nunique')
pb_marketmap['count_marketmap'] = pb_marketmap.groupby('companyid')['marketmap'].transform('nunique')

pb_marketmap = pb_marketmap[pb_marketmap['count_marketmap'] == 1]

pb_marketmap = pd.merge(pb_marketmap, pb_assignee[['companyid', 'assignee_id']], on='companyid', how='inner')

df_training = pd.merge(g_patent, pb_marketmap, on='assignee_id', how='inner')

def subsegment_name_processing(nameStr):
    nameStr = re.sub(r'[^\w\s]|_', '', nameStr)
    return nameStr.replace(" ", "")
df_training['file'] = df_training['fullname'].apply(subsegment_name_processing)
count_df_training = df_training.groupby('file').size().reset_index(name='counts_training_data')

count_df_results_and_training = count_df_results.merge(count_df_training, how='outer', on='file')
poor_sectors_df = count_df_results_and_training[(count_df_results_and_training['counts_training_data'].isnull()) | (count_df_results_and_training['counts_training_data'] < 20)]
poor_sectors_list = poor_sectors_df['file'].unique().tolist()
print(len(poor_sectors_list))
# filter out poor sectors in df_results_final
df_results_final = df_results_final[~df_results_final['file'].isin(poor_sectors_list)]

34


In [ ]:

df_results_final.to_csv(output_dir/'positive_results_100k.csv')